In [ ]:
# ============================================================================
# STEP 0: IMPORTS - What each module does
# ============================================================================

from fastapi import FastAPI           # Web framework for creating API endpoints
from pydantic import BaseModel        # Validates incoming JSON data from frontend
from agent.agentic_workflow import Graphbuilder  # YOUR LangGraph agent class
import os                             # For file operations (saving graph image)
from starlette.responses import JSONResponse  # For sending error responses

# ============================================================================
# STEP 1: CREATE FASTAPI APPLICATION INSTANCE
# ============================================================================
# This creates your API server. All endpoints will be registered on this object.
# In other projects: Replace with your app name, keep as 'app' for consistency
app = FastAPI()

# ============================================================================
# STEP 2: DEFINE REQUEST DATA MODEL (Pydantic)
# ============================================================================
# This defines WHAT data the frontend must send.
# Frontend will send JSON like: {"query": "Plan 5 days in Goa"}
# Pydantic automatically validates that 'query' field exists and is a string.
# 
# In other projects: Change class name and fields based on your needs
# Example for chatbot: class ChatRequest(BaseModel): message: str; user_id: int
class QueryRequest(BaseModel):
    query: str   # The user's travel question from Streamlit/HTML form

# ============================================================================
# STEP 3: DEFINE THE API ENDPOINT
# ============================================================================
# @app.post means this endpoint handles HTTP POST requests.
# POST is used because user is SENDING input to be processed (not just reading data)
# "/query" is the URL path. Frontend will call: http://localhost:8000/query
#
# 'async' allows handling multiple requests concurrently (non-blocking)
# 'query: QueryRequest' - FastAPI automatically parses JSON body into this Pydantic object
#
# In other projects: Change endpoint name, path, and function name as needed
# Example: @app.post("/chat") async def chat_endpoint(request: ChatRequest):
@app.post("/query")
async def query_travel_agent(query: QueryRequest):
    
    # ========================================================================
    # STEP 4: TRY BLOCK - Handle potential errors gracefully
    # ========================================================================
    # Wrapping code in try-except ensures the API returns a proper error response
    # instead of crashing the server.
    try:
        
        # ====================================================================
        # STEP 5: LOG THE INCOMING REQUEST (Debugging)
        # ====================================================================
        # Prints to console so you can see what the user asked
        # In production, use proper logging instead of print()
        print(f"Received query: {query}")
        
        # ====================================================================
        # STEP 6: CREATE THE AGENT INSTANCE
        # ====================================================================
        # Graphbuilder() creates a new instance of your LangGraph agent
        # This loads the LLM, tools, and builds the workflow.
        #
        # In other projects: Import your specific agent class
        # Note: For production, create this ONCE outside the endpoint (global)
        # to avoid recreating the agent for every request.
        graph = Graphbuilder()
        
        # ====================================================================
        # STEP 7: BUILD/COMPILE THE GRAPH
        # ====================================================================
        # graph() calls the __call__ method in Graphbuilder class
        # which internally calls build_graph() and returns the compiled graph.
        # This is the runnable LangGraph workflow.
        #
        # Alternative (if no __call__): react_app = graph.build_graph()
        react_app = graph()  # Returns compiled LangGraph
        
        # ====================================================================
        # STEP 8: (OPTIONAL) SAVE GRAPH VISUALIZATION
        # ====================================================================
        # This generates a PNG image of your agent workflow (nodes and edges)
        # Useful for debugging and documentation.
        # The image is saved in your current working directory.
        #
        # In other projects: Remove this in production if you don't need it
        png_graph = react_app.get_graph().draw_mermaid_png()
        with open("my_graph.png", "wb") as f:
            f.write(png_graph)
        print(f"Graph saved as 'my_graph.png' in {os.getcwd()}")
        
        # ====================================================================
        # STEP 9: PREPARE INPUT FOR LANGGRAPH
        # ====================================================================
        # LangGraph expects state as a dictionary with a "messages" key.
        # The "messages" value must be a LIST (even if it's one question).
        # 
        # Why a list? Because LangGraph maintains conversation history.
        # Each new message gets APPENDED to this list as the agent runs.
        #
        # In other projects: If you have custom state, create appropriate dict
        # Example: state = {"messages": [user_query], "user_id": 123}
        messages = {"messages": [query.query]}  # Note: query.query (class.field)
        
        # ====================================================================
        # STEP 10: RUN THE LANGGRAPH AGENT
        # ====================================================================
        # .invoke() runs the agent synchronously (waits for complete execution)
        # The agent goes through the ReAct loop: 
        #   START → agent → (tool calls?) → tools → agent → END
        # Returns the final state with all messages.
        #
        # For production with async: Use await react_app.ainvoke(messages)
        output = react_app.invoke(messages)
        
        # ====================================================================
        # STEP 11: EXTRACT THE FINAL ANSWER FROM RESPONSE
        # ====================================================================
        # The output dictionary contains "messages" key with list of ALL messages:
        #   [
        #     HumanMessage(user query),      # Index 0
        #     AIMessage(tool call),          # Index 1 (if tools called)
        #     ToolMessage(tool result),      # Index 2 (if tools called)
        #     AIMessage(final answer)        # Index -1 (LAST message)
        #   ]
        #
        # [-1] means "take the last item in the list"
        # .content extracts the text content from the message object
        #
        # In other projects: If you have custom state, extract accordingly
        if isinstance(output, dict) and "messages" in output:
            final_output = output["messages"][-1].content  # Last AI response
        else:
            final_output = str(output)  # Fallback for unexpected format
        
        # ====================================================================
        # STEP 12: RETURN SUCCESS RESPONSE TO FRONTEND
        # ====================================================================
        # FastAPI automatically converts this dict to JSON.
        # Frontend (Streamlit/HTML) will receive: {"answer": "Your itinerary..."}
        #
        # In other projects: Return appropriate fields for your frontend
        return {"answer": final_output}
    
    # ============================================================================
    # STEP 13: HANDLE ERRORS
    # ============================================================================
    # If ANY error occurs in try block, catch it here and return HTTP 500 error.
    # JSONResponse ensures frontend gets proper error format.
    #
    # NOTE: "ValueError" in quotes is a string, not the actual ValueError class!
    # Fix: Change to 'except Exception as e:' to catch ALL errors
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})


# ============================================================================
# BONUS: HOW TO RUN THIS FILE (Not in your code, but add as comment)
# ============================================================================
# 
# Save this file as 'api.py' and run in terminal:
#   uvicorn api:app --reload --port 8000
#
# Parameters:
#   api      = filename (api.py)
#   app      = FastAPI instance variable name (app)
#   --reload = Auto-restart on code changes (development only)
#   --port   = Which port to run on (8000 is default)
#
# Test the API with curl:
#   curl -X POST http://localhost:8000/query \
#        -H "Content-Type: application/json" \
#        -d '{"query": "Plan 5 days in Goa"}'
#
# ============================================================================
# REUSABLE TEMPLATE FOR OTHER PROJECTS
# ============================================================================
#
# To use this pattern in another project, change these parts:
#
# 1. Change import: from your_agent_file import YourAgentClass
# 2. Change Pydantic model: class YourRequest(BaseModel): your_field: str
# 3. Change endpoint path and name: @app.post("/your-path")
# 4. Change agent instantiation: agent = YourAgentClass()
# 5. Change state preparation based on your agent's expected state
# 6. Change extraction logic based on your agent's return format
# 7. Change return fields based on what frontend expects
#
# The FLOW (steps 1-13) remains the SAME for any LLM agent API!
#
# ============================================================================

# AI Trip Planner - Streamlit Frontend

## Features Implemented

### 1. Chat Interface
- Real-time chat interface with user and assistant message bubbles
- Persistent chat history within session using `st.session_state`
- Messages displayed chronologically with role-based styling (user/assistant)

### 2. Form-Based Input
- Text input with clear_on_submit for better UX
- Submit button triggers backend API call
- Loading spinner during agent processing (60-second timeout)

### 3. Backend Integration
- HTTP POST requests to FastAPI backend at `http://localhost:8000/query`
- Error handling for connection failures, timeouts, and server errors
- JSON response parsing with fallback messages

### 4. Session Management
- `st.session_state.messages` preserves conversation history across Streamlit reruns
- Initialized as empty list on first load

### 5. UI/UX Enhancements
- Custom page configuration (title, icon, wide layout)
- Sidebar with project information and sample questions
- Professional styling with emoji icons and clear visual hierarchy
- Caption and header for context

### 6. Error Resilience
- Graceful handling of:
  - Backend connection errors
  - Request timeouts (>60 seconds)
  - Non-200 HTTP responses
  - Generic exceptions

## Current Limitations

### 1. No Conversation Context for Agent
- Each query sent to backend WITHOUT previous conversation history
- Agent treats each question independently
- Example: Asking "Hotels?" after "Weather in Manali?" loses location context

### 2. No Persistent Storage
- Chat history exists only in `st.session_state`
- Page refresh (F5) clears all conversation history
- No database backup for long-term conversations

### 3. Single Session Only
- No support for multiple user sessions
- All users share same chat interface (no session isolation)

### 4. No Chat Management Features
- Cannot delete specific messages
- Cannot edit previous queries
- No conversation export functionality

### 5. No Streaming Responses
- Waits for complete agent response before displaying
- No token-by-token streaming for better UX

## Future Improvement Scope

### Immediate Improvements (High Priority)

| Improvement | Description | Benefit |
|-------------|-------------|---------|
| **Send Conversation History** | Modify backend to accept `history` parameter and send previous messages | Agent maintains context across turns |
| **Session Persistence** | Save conversations to PostgreSQL/Redis with session ID | History survives page refresh |

### Short-Term Enhancements (Medium Priority)

| Improvement | Description | Benefit |
|-------------|-------------|---------|
| **Conversation Export** | Add download button to save chat as JSON/Markdown | Users can save their trip plans |
| **Clear Chat Button** | Button to reset `st.session_state.messages` | Manual conversation reset |
| **Streaming Responses** | Use `requests.post(..., stream=True)` with SSE | Real-time token-by-token display |
| **Input Validation** | Enhanced validation with character limits and profanity filter | Better user experience |

### Long-Term Enhancements (Low Priority)

| Improvement | Description | Benefit |
|-------------|-------------|---------|
| **Multi-Session Support** | Session IDs in URL (`?session_id=123`) | Multiple parallel conversations |
| **Message Editing** | Edit previous user queries and regenerate responses | Correct mistaken inputs |
| **Voice Input** | Speech-to-text integration | Hands-free interaction |
| **Conversation Search** | Search within past conversations | Find previous trip plans |
| **User Authentication** | Login system for personalized chat history | Secure multi-user support |

## Required Code Modifications

### 1. For Agent Context (Priority Fix)

**Streamlit Change:**
```python
# Prepare history from session_state
history = [
    {"role": msg["role"], "content": msg["content"]}
    for msg in st.session_state.messages
]

response = requests.post(
    f"{BASE_URL}/query",
    json={"query": user_input, "history": history}
)

In [ ]:
import streamlit as st
import datetime
import requests
import sys

# ============================================================================
# BACKEND API ENDPOINT (Your FastAPI server)
# ============================================================================
BASE_URL = "http://localhost:8000"  # FastAPI backend endpoint

# ============================================================================
# PAGE CONFIGURATION (Must be first Streamlit command)
# ============================================================================
st.set_page_config(
    page_title="AI Trip Planner",
    page_icon="✈️",
    layout="wide"
)

# ============================================================================
# TITLE
# ============================================================================
st.title("✈️ AI Trip Planner")
st.caption("Plan your perfect vacation with AI-powered recommendations")

# ============================================================================
# INITIALIZE SESSION STATE (Preserves chat history across reruns)
# ============================================================================
if "messages" not in st.session_state:
    st.session_state.messages = []

# ============================================================================
# HEADER / SUBHEADER (Optional)
# ============================================================================
st.header("Ask me anything about your trip")

# ============================================================================
# DISPLAY CHAT HISTORY (Previous messages)
# ============================================================================
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# ============================================================================
# FORM FOR USER INPUT
# ============================================================================
with st.form(key="chat_form", clear_on_submit=True):
    user_input = st.text_input(
        "Your question:",
        placeholder="e.g., Plan a 5-day trip to Goa with ₹50000 budget"
    )
    submit_button = st.form_submit_button("Send")

# ============================================================================
# PROCESS USER INPUT WHEN SUBMITTED
# ============================================================================
if submit_button and user_input.strip():
    
    # 1. Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)
    
    # 2. Show loading spinner while calling backend
    with st.spinner("Planning your trip... 🤖"):
        try:
            # 3. Call FastAPI backend
            response = requests.post(
                f"{BASE_URL}/query",
                json={"query": user_input},
                timeout=60  # 60 seconds timeout for agent processing
            )
            
            # 4. Check if request was successful
            if response.status_code == 200:
                answer = response.json().get("answer", "No response from server")
            else:
                answer = f"Error: Server returned {response.status_code}"
                
        except requests.exceptions.ConnectionError:
            answer = "Error: Cannot connect to backend. Is FastAPI running on port 8000?"
        except requests.exceptions.Timeout:
            answer = "Error: Request timed out. Please try again."
        except Exception as e:
            answer = f"Error: {str(e)}"
    
    # 5. Add assistant response to chat history
    st.session_state.messages.append({"role": "assistant", "content": answer})
    with st.chat_message("assistant"):
        st.markdown(answer)
    
    # 6. Rerun to refresh the UI
    st.rerun()

# ============================================================================
# SIDEBAR (Optional - for additional info)
# ============================================================================
with st.sidebar:
    st.subheader("About")
    st.write("This AI Trip Planner uses:")
    st.write("- LangGraph for agentic workflow")
    st.write("- Real-time weather & place search")
    st.write("- Currency conversion & expense calculator")
    
    st.divider()
    st.subheader("Sample Questions")
    st.write("• Plan 5 days in Goa with ₹50000")
    st.write("• What's the weather in Manali?")
    st.write("• Convert 100 USD to INR")
    st.write("• Calculate daily budget for 7 days in Dubai")

# There are couple of ways to create tools

### 1. with decorator

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI(model="o4-mini",api_key=os.environ.get("OPENAI_API_KEY"))

In [4]:
llm.invoke("Hi")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 7, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'o4-mini-2025-04-16', 'system_fingerprint': None, 'id': 'chatcmpl-Doqo1OhBZUE3Dneo4csFOgjDwxbTk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eac92-afcf-7200-8ef6-f71d463a57db-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 91, 'total_tokens': 98, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})

In [5]:
from langchain.tools import tool
@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two integers.

    Args:
        a (int): The first integer.
        b (int): The second integer.

    Returns:
        int: The product of a and b.
    """
    return a * b

In [6]:
multiply

StructuredTool(name='multiply', description='Multiply two integers.\n\nArgs:\n    a (int): The first integer.\n    b (int): The second integer.\n\nReturns:\n    int: The product of a and b.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x110cd3eb0>)

### 2. with StructuredTool - it gives more scope for customisation

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel,Field

In [ ]:
class WeatherInput(BaseModel):
    city: str

def get_weather(city: str) -> str:
    """
    Get the weather for a given city.

    Args:
        city (str): The name of the city.

    Returns:
        str: A string describing the weather in the city.
    """
    return f"The weather in {city} is sunny."

weather_tool = StructuredTool.from_function(
    func=get_weather,
    name="get_weather",
    description="Fetches real-time weather data for a city",
    args_schema=WeatherInput,  # Pydantic object for data validation
)

### 3.

In [ ]:
class WeatherInput(BaseModel):
    city: str = Field(..., description="City name")
    units: str = Field("metric", description="metric or imperial")

class GetWeatherTool(StructuredTool):
    name: ClassVar[str] = "get_weather"           
    description: ClassVar[str] = (
        "Fetches weather data for a city"
    )
    args_schema: ClassVar[Type[BaseModel]] = WeatherInput

    def _run(self, city: str, units: str = "metric") -> str:
        return get_weather(city, units)